# Teaching an LLM to Solve Math Problems with RL on Amazon SageMaker Training Sessions

## Part 1: Train

This is part 1 of the walkthrough. Here we:

- Create a live, GPU-backed **Training Session** running a base model with LoRA adapters
- Measure the base model's accuracy on held-out GSM8K problems (baseline)
- Write a small [GRPO](https://arxiv.org/abs/2402.03300)-style RL loop from scratch: sample
  responses, score them, turn rewards into advantages, and update the model
- Measure accuracy again after training, and save a checkpoint

**The three SDK clients you'll use:**

| Client | Role |
|---|---|
| `ServiceClient` | Entry point — holds the platform-level settings (role, model package group, S3 output, optional KMS key / VPC) and creates sessions |
| `TrainingClient` | Drives the loop — `forward_backward`, `optim_step`, `save_weights_for_sampler`, `save_state` |
| `SamplingClient` | Ask the current model a question and get an answer |

**Every operation is asynchronous.** Training and sampling calls return an `APIFuture`
*immediately* — the request is queued and shipped by a background submitter, so thousands of
requests can be in flight without one round trip per call. Call `.result()` on the future when
you need the value (or `await` it in async code). That's what lets a step submit all of
its rollouts at once, and submit its `optim_step` behind its `forward_backward`.

SageMaker owns
the model's forward/backward computation, GPU allocation, and LoRA application. You own: *when*
to call each primitive, *which* built-in loss function to use (`ppo`, `cross_entropy`,
`importance_sampling`, `cispo`), and *what per-token signals* (`advantages`, `logprobs`,
`weights`) feed into it — that last part is how your own reward/advantage logic (Section 7
below) gets injected into someone else's loss implementation. Checkpointing
(`save_weights_for_sampler`, `save_state`) and sampling (`sample`) happen on your schedule too,
driven by this Python process — there's no config file telling the service what to do next.

> **Prerequisites:** run `infrastructure/setup_infra.py` (creates the IAM role)

## 1. Install the SDK

`requirements.txt` pins `sagemaker>=3.25.0`, the first SageMaker Python SDK release that
includes the Training Sessions primitives (`sagemaker.train.training_session`).


In [ ]:
%pip install -r requirements.txt -U -q

## 2. Load config

No config file to read -- region, account, execution role, and default S3 bucket are all
discovered at runtime via the SageMaker session helpers, the same way a SageMaker Studio
notebook would. `get_execution_role()` works automatically inside SageMaker; outside of it
(e.g. a local kernel), it falls back to looking up the role `setup_infra.py` created by name.

These values feed the `ServiceClient` we build in Section 3, once the model package group it
should register checkpoints into exists.

In [ ]:
import boto3
from botocore.exceptions import ClientError
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.train.training_session import ServiceClient

sess = Session()

account_id = sess.account_id()
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
REGION = sess.boto_region_name

ROLE_NAME = "SageMakerTrainingSessionRole"  # created by infrastructure/setup_infra.py


def resolve_execution_role() -> str:
    """The role the session assumes. Prefer ROLE_NAME: it carries the trust policy the
    service needs. Studio space roles often cannot call iam:GetRole, hence the ARN
    fallback; get_execution_role() is last because outside SageMaker it returns the
    caller's role instead of raising."""
    try:
        return boto3.client("iam").get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    except ClientError as err:
        code = err.response["Error"]["Code"]
        if code in ("AccessDenied", "AccessDeniedException"):
            print(f"iam:GetRole denied; using the conventional ARN for {ROLE_NAME}")
            return f"arn:aws:iam::{account_id}:role/{ROLE_NAME}"
        if code in ("NoSuchEntity", "NoSuchEntityException"):
            print(
                f"{ROLE_NAME} not found -- run infrastructure/setup_infra.py. "
                "Falling back to get_execution_role(), which must itself be trusted by "
                "training-session.sagemaker.amazonaws.com."
            )
            return get_execution_role()
        raise


role = resolve_execution_role()

MODEL_PACKAGE_GROUP_NAME = "gsm8k-grpo-training-sessions"

if default_prefix:
    OUTPUT_S3_PATH = f"s3://{bucket_name}/{default_prefix}/outputs/training-sessions"
else:
    OUTPUT_S3_PATH = f"s3://{bucket_name}/outputs/training-sessions"

print(f"account id: {account_id}")
print(f"role arn:   {role}")
print(f"bucket:     {bucket_name}")
print(f"region:     {REGION}")

## 3. Create the Model Package Group, then the ServiceClient

Training checkpoints (`save_state` / `save_weights_for_sampler`) are registered as versions
inside a **Model Package Group** — think of it as a folder for this project's checkpoints.
Unlike the IAM role, this doesn't need elevated permissions, so we create it here rather than in
the one-time infra setup step. Get-or-create, so it's safe to re-run this notebook.

In [ ]:
import hashlib
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

MAX_MPG_NAME_LENGTH = 63
if len(MODEL_PACKAGE_GROUP_NAME) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(MODEL_PACKAGE_GROUP_NAME.encode()).hexdigest()[:6]
    keep = MAX_MPG_NAME_LENGTH - len(digest) - 1  # room for the "-<digest>" suffix
    MODEL_PACKAGE_GROUP_NAME = f"{MODEL_PACKAGE_GROUP_NAME[:keep].rstrip('-')}-{digest}"

# Get-or-create via the typed sagemaker.core resource (the same SDK sm-training_sessions_2_deploy_grpo_checkpoint.ipynb uses), so it's
# safe to re-run this notebook. Passing the group explicitly below keeps this experiment's
# checkpoints together -- left unset, the SDK would use the account's shared default group.
try:
    model_package_group = ModelPackageGroup.get(model_package_group_name=MODEL_PACKAGE_GROUP_NAME)
    print(f"Model Package Group already exists: {MODEL_PACKAGE_GROUP_NAME}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        model_package_group_description="GSM8K RL checkpoints -- Training Sessions walkthrough",
    )
    print(f"Created Model Package Group: {MODEL_PACKAGE_GROUP_NAME}")

# Deterministic from account_id + REGION + the (length-safe) name above, so there's nothing to
# persist: sm-training_sessions_2_deploy_grpo_checkpoint.ipynb and infrastructure/cleanup_infra.py rebuild this same ARN from the same
# MODEL_PACKAGE_GROUP_NAME constant.
MODEL_PACKAGE_GROUP_ARN = (
    f"arn:aws:sagemaker:{REGION}:{account_id}:model-package-group/{MODEL_PACKAGE_GROUP_NAME}"
)
print("Model Package Group ARN:", MODEL_PACKAGE_GROUP_ARN)

# Platform-level settings, set once here and inherited by every session this client creates.
svc = ServiceClient(
    region=REGION,
    role_arn=role,
    model_package_group_arn=MODEL_PACKAGE_GROUP_ARN,
    s3_output_path=OUTPUT_S3_PATH,
)

## 4. Prepare the data

We download **GSM8K** (grade-school math word problems) from HuggingFace and extract each
problem's final numeric answer from its `#### <number>` format. To keep the walkthrough fast we
take a small train/eval split (bump `NUM_TRAIN` / `NUM_EVAL` below for a longer run -- the prep
step rebuilds automatically when the cached files no longer match those counts — GSM8K has ~7.5k train / ~1.3k test problems). The **eval** set is drawn from
GSM8K's held-out *test* split, so it measures accuracy on problems the model never trains on.

Each row we write to `data/train.jsonl` / `data/eval.jsonl` is one problem:
`{"question": ..., "ground_truth": ...}`.

In [ ]:
from datasets import load_dataset
import json
import os
import re


# GSM8K answers end with "#### <number>"; pull out that final numeric answer as the ground truth.
def extract_ground_truth(answer: str) -> str:
    match = re.search(r"####\s*([\-0-9\.,]+)", answer)
    return match.group(1).replace(",", "").strip() if match else ""


NUM_TRAIN = 40  # training problems (from GSM8K's train split)
NUM_EVAL = 30  # held-out eval problems (from GSM8K's test split)
FORCE_PREPARE = False  # set True to re-download and overwrite data/*.jsonl

os.makedirs("data", exist_ok=True)


# Cache on the row counts, not just existence: bumping NUM_TRAIN / NUM_EVAL above would
# otherwise be silently ignored because the old files are still sitting there.
def row_count(path):
    if not os.path.exists(path):
        return -1
    with open(path) as f:
        return sum(1 for _ in f)


cached = row_count("data/train.jsonl") == NUM_TRAIN and row_count("data/eval.jsonl") == NUM_EVAL

if not FORCE_PREPARE and cached:
    print(
        f"data/train.jsonl ({NUM_TRAIN}) and data/eval.jsonl ({NUM_EVAL}) already match "
        "the requested sizes -- skipping prep."
    )
else:
    print("Downloading GSM8K from HuggingFace (openai/gsm8k)...")
    gsm8k = load_dataset("openai/gsm8k", "main")

    def to_rows(split, n):
        subset = split.select(range(min(n, len(split))))
        return [
            {"question": ex["question"], "ground_truth": extract_ground_truth(ex["answer"])}
            for ex in subset
        ]

    # Train from the train split; eval from the held-out test split (never trained on).
    for path, rows in [
        ("data/train.jsonl", to_rows(gsm8k["train"], NUM_TRAIN)),
        ("data/eval.jsonl", to_rows(gsm8k["test"], NUM_EVAL)),
    ]:
        with open(path, "w") as f:
            for r in rows:
                f.write(json.dumps(r) + "\n")
        print(f"Wrote {path} ({len(rows)} problems)")

In [ ]:
import json


def load_jsonl(path):
    problems = []
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            problems.append((row["question"], row["ground_truth"]))
    return problems


train_problems = load_jsonl("data/train.jsonl")
eval_problems = load_jsonl("data/eval.jsonl")
print(f"train: {len(train_problems)}  |  eval: {len(eval_problems)}")

## 5. Create a Training Session

A **Training Session** is a live, GPU-backed session running your base model with LoRA adapters.
`ServiceClient` is the entry point; `create_lora_training_client(...)` creates the session and
blocks until it's `Running`. Only the model identity and the LoRA shape are passed here — role,
model package group, and S3 output come from the `ServiceClient` built in Section 3.

In [ ]:
print(f"Base models available in {REGION}:")
for m in svc.list_base_models():
    print(" -", m)

In [ ]:
import time

BASE_MODEL = "openai/gpt-oss-20b"  # a name from svc.list_base_models()
SESSION_NAME = f"gsm8k-grpo-{int(time.time())}"

training_client = svc.create_lora_training_client(
    BASE_MODEL,
    training_session_name=SESSION_NAME,
    rank=32,  # LoRA rank -- higher = more trainable capacity, more memory
    alpha=64,  # LoRA scaling factor, conventionally 2x rank
    # model_package_group_arn=...,   # optional: override the ServiceClient default
    # s3_output_path=...,            # optional: override for this session only
    # kms_key_arn=...,   # encrypt saved weights/checkpoints with your own key
    # vpc_config={"subnets": [...], "security_group_ids": [...]},
)
print("Session ready:", training_client.training_session_name)

## 6. Baseline: how good is the base model already

Before training, let's see how the untrained model does. We need a **tokenizer** to turn text
into token IDs, and a **SamplingClient** to ask the model a question -- we'll reuse both, along with the `build_prompt()`, `compute_reward()`, and
`SAMPLING_PARAMS` defined below, in Section 7's rollout collection -- same prompt
template and scoring rule for training as for eval.

In [ ]:
def build_prompt(question: str) -> str:
    """Render the question in the GPT-OSS harmony chat format, and ask for step-by-step
    reasoning with the final answer after '#### ' so the reward function can parse it.
    Prompting the model in its trained format is what makes it emit the marker (and stop)
    reliably -- a raw 'Question:/Answer:' string does not.

    This template is specific to GPT-OSS. Changing BASE_MODEL means rewriting it for
    that model's chat format (or building it with the tokenizer's chat template);
    left as-is, the harmony control tokens become literal text and the model will
    neither reason nor emit the '#### ' marker the reward function looks for."""
    instruction = (
        f"{question}\n\n"
        "Solve this step by step, then give the final numeric answer on its own "
        'line after "#### ".'
    )
    return (
        "<|start|>system<|message|>You are a helpful assistant.\n"
        "Reasoning: high\n"
        "# Valid channels: analysis, commentary, final. "
        "Channel must be included for every message.<|end|>"
        f"<|start|>user<|message|>{instruction}<|end|>"
        "<|start|>assistant"
    )

### Reward function

We'll reuse this in both evaluation and training: GSM8K answers end with `#### <number>`. A
response is correct (reward `1.0`) if the number it outputs after `####` matches the ground
truth, otherwise `0.0`.

In [ ]:
import re


def extract_answer(text: str) -> str | None:
    # The final answer lives at the end of the response, so look only at the tail and
    # take the LAST "#### <number>" match -- robust to the model reasoning with "####"
    # earlier or repeating the answer.
    tail = text[-300:]
    solutions = re.findall(r"####\s*\$?\s*(-?[0-9\.,]+)", tail)
    if not solutions:
        return None
    return solutions[-1].replace(",", "").strip()


def compute_reward(response_text: str, ground_truth: str) -> float:
    predicted = extract_answer(response_text)
    return 1.0 if predicted == ground_truth else 0.0

Now sample one response per eval problem and score it. `sample()` hands back an `APIFuture`
without waiting, so we submit **every** problem first and only then resolve the futures — the
whole eval set generates concurrently instead of one problem per round trip.

In [ ]:
from sagemaker.train.training_session import SamplingParams

tokenizer = training_client.get_tokenizer()
sampling_client = training_client.create_sampling_client()

SAMPLING_PARAMS = SamplingParams(max_tokens=512, temperature=0.7, top_p=0.9)

# Sampling is stochastic, so one response per problem measures the dice as much as the
# model: 30 problems x 1 sample carries a ~9 point standard error, wider than anything
# 10 training steps will move. Averaging a few samples per problem shrinks that, and
# evaluate() prints the error bar so you can tell a real gap from noise. Greedy decoding
# would be the cheaper way to cut variance where the service supports it.
EVAL_SAMPLES_PER_PROBLEM = 4


def evaluate(problems, label=""):
    """Sample EVAL_SAMPLES_PER_PROBLEM responses per problem and report mean accuracy."""
    # Submit everything first (non-blocking), then resolve in order.
    futures = [
        sampling_client.sample(
            tokenizer.encode(build_prompt(question), add_special_tokens=False),
            num_samples=EVAL_SAMPLES_PER_PROBLEM,
            sampling_params=SAMPLING_PARAMS,
        )
        for question, _ in problems
    ]

    scores = []
    for (_, ground_truth), future in zip(problems, futures):
        for sample in future.result().sequences:  # blocks until this problem is done
            scores.append(compute_reward(tokenizer.decode(sample.tokens), ground_truth))

    n = len(scores)
    accuracy = sum(scores) / n
    std_err = (accuracy * (1 - accuracy) / n) ** 0.5  # binomial standard error
    print(f"{label} accuracy: {accuracy:.0%} +/- {std_err:.0%} " f"({sum(scores):.0f}/{n} samples)")
    return accuracy


baseline_accuracy = evaluate(eval_problems, label="Baseline (before training)")

## 7. Building blocks of the RL loop

Now the RL part. Each training step:
1. Pick a batch of questions
2. Sample **K responses per question** (a "group") — some will be right, some wrong
3. Turn rewards into **advantages**: how much better/worse was each response than others in
   its group?
4. Package everything into the token-level format the training API expects
5. Run one `forward_backward` + `optim_step`

This is a simplified version of **GRPO** (Group Relative Policy Optimization), minus the
parallel-rollout machinery, to keep the loop easy to follow.

### 7.1 Sampling K responses per question

One request per question, not K: `sample(..., num_samples=K)` returns all K sequences from a
single call. The helpers below reuse `build_prompt()`, `compute_reward()`, and `SAMPLING_PARAMS`
from Section 6 as-is — same prompt template and scoring rule as eval.

They're split in two so the training loop can submit every question in a batch before waiting on
any of them: `submit_group()` returns as soon as the request is queued, and `score_group()` is
where the blocking `.result()` happens.

In [ ]:
RESPONSES_PER_PROMPT = 8  # K -- how many attempts per question


def submit_group(question: str):
    """Queue ONE request asking for K responses to this question. Does not block."""
    prompt_tokens = tokenizer.encode(build_prompt(question), add_special_tokens=False)
    future = sampling_client.sample(
        prompt_tokens,
        num_samples=RESPONSES_PER_PROMPT,
        sampling_params=SAMPLING_PARAMS,
    )
    return prompt_tokens, future


def score_group(prompt_tokens, future, ground_truth: str):
    """Resolve a submitted request and score each of its K responses."""
    group = []
    for sample in future.result().sequences:
        response_text = tokenizer.decode(sample.tokens)
        group.append(
            {
                "prompt_tokens": prompt_tokens,
                "response_tokens": sample.tokens,
                "response_logprobs": sample.logprobs,
                "reward": compute_reward(response_text, ground_truth),
            }
        )
    return group

### 7.2 Turning rewards into advantages

If all K responses in a group get the same reward (all right or all wrong), there's nothing to
learn from — so we center each response's reward against its group's average. A response that
did *better than its group* gets a positive advantage (reinforce it); *worse than its group*
gets a negative one (discourage it).

In [ ]:
import numpy as np


def compute_advantages(group):
    """Center (and normalize) rewards within a group -- classic GRPO baseline."""
    rewards = np.array([r["reward"] for r in group])
    advantages = rewards - rewards.mean()
    std = rewards.std()
    if std > 1e-8:
        advantages = advantages / std
    for response, advantage in zip(group, advantages):
        response["advantage"] = float(advantage)
    return group

### 7.3 Building the training datum

The training API works on **tokens**, not text. For every response we build one *datum*: the
input tokens (prompt + response), the target tokens to predict, the log-probabilities the policy
had *when it sampled* the response, the advantage, and a `weights` mask marking which tokens are
the response — the ones we train on (the prompt is masked out with `0`s).

In [ ]:
def build_training_datum(response: dict) -> dict:
    # Next-token prediction shifts inputs and targets by one, so the prompt occupies
    # len(prompt_tokens) - 1 positions of the shifted sequence: the prompt's final token
    # is an input whose target is the response's first token -- the first position we
    # train on.
    prompt_len = len(response["prompt_tokens"]) - 1
    tokens = response["prompt_tokens"] + response["response_tokens"]

    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    response_len = len(input_tokens) - prompt_len

    logprobs = [0.0] * prompt_len + list(response["response_logprobs"])
    advantages = [0.0] * prompt_len + [response["advantage"]] * response_len
    weights = [0.0] * prompt_len + [1.0] * response_len  # 0 = prompt, 1 = response

    seq_len = len(input_tokens)
    return {
        "modelInput": {"chunks": [{"tokens": input_tokens, "type": "encoded_text"}]},
        "lossFnInputs": {
            "target_tokens": {"data": target_tokens, "dtype": "int64", "shape": [seq_len]},
            "logprobs": {"data": logprobs, "dtype": "float32", "shape": [seq_len]},
            "advantages": {"data": advantages, "dtype": "float32", "shape": [seq_len]},
            "weights": {"data": weights, "dtype": "float32", "shape": [seq_len]},
        },
    }

### 7.4 One training step

With a batch of datums in hand, a training step is just two calls: `forward_backward` computes
gradients from the loss function (`"ppo"`, here — the other built-in options are
`cross_entropy`, `importance_sampling`, and `cispo`), and `optim_step` applies them.

Both return an `APIFuture` right away. The SDK stamps a sequence number at call time, so the
optimizer step can be *submitted* before the forward/backward has resolved and still run after
it — the two pipeline instead of costing two serial round trips. We resolve both futures at
the end of the step because we want the metrics; a loop that didn't could keep going.

Because they are separate calls, you can call `forward_backward` several times to accumulate
gradients across microbatches before a single `optim_step`, which sets the effective batch
size.

There's also a forward-only `forward(batch, loss_fn=...)` (no gradient, no weight update) for
getting logprobs/loss to *score* a batch without training on it — not used in this notebook,
but useful for perplexity-style evaluation.

In [ ]:
from sagemaker.train.training_session import AdamParams

ADAM_PARAMS = AdamParams(learning_rate=1e-5)


def train_step(datums: list) -> dict:
    # Submit both, then wait: the optim step is sequenced behind the forward/backward
    # by the SDK, so it doesn't have to wait for the gradients to come back here.
    fwd_bwd_op = training_client.forward_backward(datums, loss_fn="ppo")
    optim_op = training_client.optim_step(ADAM_PARAMS)

    result = fwd_bwd_op.result()
    optim_op.result()

    # The PPO diagnostics come back in result.metrics: actor/pg_loss, actor/ppo_kl,
    # actor/pg_clipfrac, actor/pg_clipfrac_lower, actor/grad_norm. The same dict also
    # carries throughput counters (estimated_flops, global_token_num, mfu,
    # promised_flops).
    metrics = result.metrics or {}
    kept = {k: v for k, v in metrics.items() if k.startswith("actor/")}
    # Keep the token counter too: you are billed for tokens processed rather than for
    # wall-clock time, so this is the closest thing to a per-step cost readout.
    if "global_token_num" in metrics:
        kept["tokens"] = metrics["global_token_num"]
    return kept

## 8. Put it all together: the training loop

For each step: sample a batch of questions from the training pool, submit all of their rollout
requests, score them as they come back, turn rewards into advantages, build the datums, and
train. Because the submits don't block, the whole step's `PROMPTS_PER_STEP x K` responses are
generated concurrently.

Each step ends by refreshing the sampler so the *next* step's rollouts come from the policy we
just trained — that's what keeps the loop on-policy.

In [ ]:
import random

NUM_STEPS = 10  # number of training steps (weight updates) to run
PROMPTS_PER_STEP = (
    4  # questions per step -> 4 questions x K=8 responses = 32 training examples/step
)
SAVE_STATE_EVERY = 5  # checkpoint cadence, in steps (0 disables)

# Shuffled "deck" of questions we draw PROMPTS_PER_STEP at a time from.
train_pool = list(train_problems)
random.shuffle(train_pool)

for step in range(NUM_STEPS):
    # Cycle through the training pool; reshuffle once we've used it all (a soft "epoch
    # boundary" -- every question gets seen ~equally often). With the defaults this never
    # fires, since NUM_STEPS x PROMPTS_PER_STEP is exactly one pass over train_problems;
    # raise NUM_STEPS and it starts wrapping.
    if len(train_pool) < PROMPTS_PER_STEP:
        train_pool = list(train_problems)
        random.shuffle(train_pool)
    batch = [train_pool.pop() for _ in range(PROMPTS_PER_STEP)]  # draw this step's questions

    start = time.time()
    datums = []  # token-level training examples for this step, fed to train_step()
    rewards_this_step = []  # every individual right/wrong score this step, for batch_accuracy below
    n_degenerate = 0  # groups skipped because every response scored the same

    # Queue every question's rollout request up front -- each returns an APIFuture
    # immediately, so all PROMPTS_PER_STEP x K responses generate concurrently.
    pending = [(ground_truth, submit_group(question)) for question, ground_truth in batch]

    for ground_truth, (prompt_tokens, future) in pending:
        # Wait for this question's K attempts and score each one.
        group = score_group(prompt_tokens, future, ground_truth)
        rewards_this_step.extend(r["reward"] for r in group)
        # Skip degenerate groups: if every response earned the same reward (all right or
        # all wrong), the centered advantages are all zero -- no gradient signal, so
        # there's nothing to learn from this question.
        if len({r["reward"] for r in group}) == 1:
            n_degenerate += 1
            continue
        # Turn this question's K raw rewards into GRPO advantages (centered on their own group).
        group = compute_advantages(group)
        # Convert each scored attempt into the token-level datum format and queue it for training.
        datums.extend(build_training_datum(r) for r in group)

    # accuracy = fraction of all attempts this step that were exactly correct.
    accuracy = sum(rewards_this_step) / len(rewards_this_step)
    elapsed = time.time() - start

    # Every group was degenerate -> no training examples this step; skip the update.
    if not datums:
        print(
            f"step {step:>2}  batch_accuracy={accuracy:.0%}  "
            f"degenerate={n_degenerate}/{PROMPTS_PER_STEP}  (no update)  ({elapsed:.1f}s)"
        )
        continue

    # The actual weight update: one forward_backward over the batch, then one optim_step.
    metrics = train_step(datums)

    # Refresh the sampler so the NEXT step's rollouts come from the policy we just trained:
    # this saves the updated weights and hands back a SamplingClient bound to them. Rebinding
    # the module-level `sampling_client` is what puts submit_group()/evaluate() on the new policy.
    sampling_client = training_client.save_weights_and_get_sampling_client(
        description=f"policy after step {step}"
    )

    # Checkpoint periodically. A throttled call, a preempted session, or a lost kernel
    # ends this loop, and the session cannot be reattached -- without a save_state you
    # restart from the base model. Resuming from the newest checkpoint is the recovery
    # path (Section 11). This blocks, so it costs a few seconds per checkpoint.
    if SAVE_STATE_EVERY and (step + 1) % SAVE_STATE_EVERY == 0 and step + 1 < NUM_STEPS:
        training_client.save_state(description=f"step {step} checkpoint").result()
        print(f"     checkpointed at step {step}")

    metrics_str = "  ".join(
        f"{k}={v:,d}" if isinstance(v, int) else f"{k}={v:.4g}" for k, v in metrics.items()
    )
    print(
        f"step {step:>2}  batch_accuracy={accuracy:.0%}  "
        f"degenerate={n_degenerate}/{PROMPTS_PER_STEP}  {metrics_str}  ({elapsed:.1f}s)"
    )

## 9. Evaluate again after training

Run the exact same evaluation as the baseline, on the same held-out problems — `evaluate()`
picks up the `sampling_client` the loop last rebound, i.e. the trained policy.

> **⚠️ Run Sections 9 → 10 → 11 in order, top to bottom.** `sampling_client` talks to the
> session created in Section 5. If you jump ahead and run Section 11 (`training_client.stop()`)
> before coming back here, this cell will fail with `SessionNotActiveError` — a stopped session
> can't be reused. If that happens, see the recovery snippet in Section 11.

In [ ]:
trained_accuracy = evaluate(eval_problems, label="After training")

print()
print(f"Baseline : {baseline_accuracy:.0%}")
print(f"Trained  : {trained_accuracy:.0%}")
print(f"Delta    : {trained_accuracy - baseline_accuracy:+.0%}")

## 10. Save a deployable checkpoint

The `save_weights_and_get_sampling_client()` calls inside the loop registered the **LoRA
adapter** at each step (and rebound the sampler to it). To get a self-contained model to deploy,
call `save_weights_for_sampler(merge_weights=True)` — it folds the adapter into the base model
and registers a single **merged** model package, which is what `sm-training_sessions_2_deploy_grpo_checkpoint.ipynb` deploys
(server-side merge, adds a few minutes). `save_state()` additionally saves optimizer state so a
later run can resume from it; it's not deployable on its own.

Both are submitted as futures like everything else, so `.result()` is where we wait for the ARN.
`.result()` waits indefinitely by default — pass `timeout=` seconds if you'd rather it gave up.

In [ ]:
# Merged, self-contained model for deployment (sm-training_sessions_2_deploy_grpo_checkpoint.ipynb picks up the latest one).
deployable = training_client.save_weights_for_sampler(
    description="merged model, end of run", merge_weights=True
).result()
print(
    "Deployable merged checkpoint (save_weights_for_sampler, merged):", deployable.model_package_arn
)

# Resumable checkpoint (weights + optimizer state) so you can continue training later.
state_checkpoint = training_client.save_state(description="end of run, resumable").result()
print("Resumable checkpoint (save_state):", state_checkpoint.model_package_arn)

## 11. Clean up

Stop the session when you're done — you're billed for the tokens your operations process rather
than for session wall-clock time, but a live session still holds resources, so stop it once
you're finished. The checkpoints above live independently in the Model Package Group, so
stopping the session doesn't affect them.

The SDK also stops a session automatically once the last client referencing it is garbage
collected (e.g. the cell that created it is re-run, or the kernel shuts down), so a forgotten
session doesn't stay `Running` indefinitely. That's a backstop, not a substitute for stopping
it yourself — it fires whenever the interpreter happens to collect, and never if the process
is killed outright.

**Stopping is irreversible** — once you run the cell below, `training_client` and
`sampling_client` stop working, and Section 9's evaluate cell will raise
`SessionNotActiveError` if you go back and re-run it.

**To train more after stopping, resume from a checkpoint, not from the session.** Reattaching to
a live session is no longer supported; the recommended path is to create a *new* session from a
saved model package — which also means a crashed kernel or an expired session costs you
nothing but the time since your last `save_state`:

```python
training_client = svc.create_training_client_from_state(
    model_package_arn=state_checkpoint.model_package_arn,  # from Section 10 (save_state)
    training_session_name=f"gsm8k-grpo-resumed-{int(time.time())}",
    rank=32,
    alpha=64,
)
tokenizer = training_client.get_tokenizer()
sampling_client = training_client.create_sampling_client()
```

`base_model` is read from the checkpoint's metadata when omitted, and role / model package group
/ S3 output come from `svc` as usual. This restores **weights only** — the optimizer resets.
Use `create_training_client_from_state_with_optimizer(...)` to bring Adam's state back too.

Then re-run whichever cell you needed (e.g. Section 9's `evaluate(...)`).

In [ ]:
training_client.stop()  # drains anything still queued, then stops the session
print("Session stopped.")

### Cleanup pending sessions

In [ ]:
import time

# list_sessions returns TrainingSessionSummary objects (attributes, not dict keys) and takes
# filters. Scope to this notebook's own sessions by name so a shared account's other sessions
# are left alone. "Failed" and "Expired" are already terminal, so only target the stoppable.
sessions = svc.list_sessions(name_contains="gsm8k-grpo")
TERMINAL = ("Stopped", "Stopping", "Failed", "Expired")
active = [s for s in sessions if s.status not in TERMINAL]

print(f"{len(sessions)} session(s) matching 'gsm8k-grpo', {len(active)} still active:")
for s in active:
    print(f"  {s.training_session_arn.split('/')[-1]:<45} {s.status}")

# StopTrainingSession is rate-limited (429 ThrottlingException, retryable) -- calling
# it back-to-back for every session with no delay bursts past that limit. Retry with
# backoff, and pace successive calls.
for session in active:
    name = session.training_session_arn.split("/")[-1]
    for attempt in range(5):
        try:
            svc.stop_session(training_session=name)  # name or ARN; idempotent
            print(f"Stopped: {name}")
            break
        except Exception as e:
            if "Throttling" in str(e) and attempt < 4:
                wait = 2**attempt
                print(f"  Throttled, retrying {name} in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Failed to stop {name}: {e}")
                break
    time.sleep(1)  # pace successive calls even on success